## 1. Imports & Setup

In [ ]:
import os
import sys
import json
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score, f1_score,
    precision_score, recall_score,
    classification_report
)
from tqdm.notebook import tqdm

# ── Reproducibility ───────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Device ────────────────────────────────────────────────────────
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
print(f'Using device: {DEVICE}')

# ── Paths ─────────────────────────────────────────────────────────
NOTEBOOK_DIR  = os.getcwd()
PROJECT_ROOT  = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
NASBENCH_PATH = os.path.join(PROJECT_ROOT, 'nas-bench-nlp-release')
RESULTS_PATH  = os.path.join(PROJECT_ROOT, 'results')
MODELS_PATH   = os.path.join(PROJECT_ROOT, 'models')

sys.path.insert(0, NASBENCH_PATH)
os.makedirs(MODELS_PATH, exist_ok=True)

print(f'Project root  : {PROJECT_ROOT}')
print(f'Results path  : {RESULTS_PATH}')

## 2. Load NAS Architecture

In [ ]:
# ── Load NAS-found architecture ──────────────────────────────────
ACTIVATION_OPS = ['activation_tanh', 'activation_sigm', 'activation_leaky_relu']

# Fallback: a validated GRU-like recipe used only if the JSON recipe has no
# activation ops (pure linear recurrence → NaN over 128 timesteps).
FALLBACK_RECIPE = {
    "node_0":  {"op": "linear",           "input": ["x", "h_prev_0"]},
    "node_1":  {"op": "activation_tanh",  "input": ["node_0"]},
    "node_2":  {"op": "linear",           "input": ["node_1", "h_prev_0"]},
    "node_3":  {"op": "activation_sigm",  "input": ["node_2"]},
    "node_4":  {"op": "elementwise_prod", "input": ["node_1", "node_3"]},
    "h_new_0": {"op": "linear",           "input": ["node_4", "h_prev_0"]}
}

arch_path = os.path.join(RESULTS_PATH, 'best_architecture.json')
with open(arch_path) as f:
    search_results = json.load(f)

best_arch     = search_results['best_architecture']
loaded_recipe = best_arch['recipe']

# Activation check: every value must be a plain dict with 'op'
has_activation = any(
    isinstance(v, dict) and v.get('op') in ACTIVATION_OPS
    for v in loaded_recipe.values()
)

if has_activation:
    BEST_RECIPE   = loaded_recipe
    BEST_INDEX    = best_arch['index']
    BEST_HC       = best_arch['hidden_covariance']
    RECIPE_SOURCE = 'NB04 v9 search result'
else:
    print('⚠️  Loaded recipe has no activation ops — using validated fallback.')
    BEST_RECIPE   = FALLBACK_RECIPE
    BEST_INDEX    = best_arch.get('index', -1)
    BEST_HC       = best_arch.get('hidden_covariance', 0.0)
    RECIPE_SOURCE = 'validated GRU-like fallback'

act_ops = [v['op'] for v in BEST_RECIPE.values()
           if isinstance(v, dict) and v.get('op') in ACTIVATION_OPS]

print('='*55)
print('  NAS Architecture')
print('='*55)
print(f'Source         : {RECIPE_SOURCE}')
print(f'Index          : {BEST_INDEX}')
print(f'HC score       : {BEST_HC:.4f}')
print(f'Recipe nodes   : {list(BEST_RECIPE.keys())}')
print(f'Activation ops : {act_ops}')


## 3. Load SOLD Dataset & Vocabulary

In [ ]:
# ── Load vocab from notebook 01 ───────────────────────────────────
with open(os.path.join(RESULTS_PATH, 'vocab.json'), 'r', encoding='utf-8') as f:
    word2idx = json.load(f)

VOCAB_SIZE = len(word2idx)
PAD_IDX    = word2idx['<PAD>']
UNK_IDX    = word2idx['<UNK>']
print(f'Vocabulary size: {VOCAB_SIZE}')

# ── Load SOLD ─────────────────────────────────────────────────────
print('Loading SOLD dataset...')
sold_train_raw = load_dataset('sinhala-nlp/SOLD', split='train')
sold_test_raw  = load_dataset('sinhala-nlp/SOLD', split='test')
print(f'Train: {len(sold_train_raw)} | Test: {len(sold_test_raw)}')

## 4. DataLoaders

In [ ]:
def tokenize(text):
    if not isinstance(text, str):
        return []
    return text.strip().split()

def encode_text(text, word2idx, max_len=128):
    tokens  = tokenize(text)[:max_len]
    indices = [word2idx.get(tok, UNK_IDX) for tok in tokens]
    indices = indices + [PAD_IDX] * (max_len - len(indices))
    return indices

def label_to_int(label):
    mapping = {'NOT': 0, 'OFF': 1}
    if isinstance(label, int):
        return label
    return mapping.get(str(label).upper(), 0)

class SOLDDataset(Dataset):
    def __init__(self, hf_dataset, word2idx, max_len=128):
        self.texts    = hf_dataset['text']
        self.labels   = hf_dataset['label']
        self.word2idx = word2idx
        self.max_len  = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = encode_text(self.texts[idx], self.word2idx, self.max_len)
        y = label_to_int(self.labels[idx])
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

MAX_LEN    = 128
BATCH_SIZE = 64

train_dataset = SOLDDataset(sold_train_raw, word2idx, MAX_LEN)
test_dataset  = SOLDDataset(sold_test_raw,  word2idx, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

## 5. Model


In [ ]:
try:
    from custom_rnn import CustomRNN
    print('CustomRNN imported successfully')
except ImportError as e:
    print(f'ERROR: {e}')
    raise


class NASClassifier(nn.Module):
    """
    Bidirectional NAS classifier using CustomRNN step-by-step.

    Design:
      - Forward RNN  : left→right over the sequence
      - Backward RNN : right→left (flip, run, flip back)
      - Per-timestep LayerNorm on hidden states — bounds magnitudes for
        ungated cells that lack sigmoid/tanh to self-regulate
      - Mean + Max pooling over time → richer features without bidoubling params
      - classifier: Linear(hidden_dim*4 → 2)
        (hidden_dim*2 from fwd mean+max, hidden_dim*2 from bwd mean+max)

    Matches baselines (embed_dim=128, hidden_dim=256, dropout=0.3).
    """
    def __init__(
        self,
        vocab_size,
        recipe,
        embed_dim   = 128,
        hidden_dim  = 256,
        num_classes = 2,
        dropout     = 0.3,
        pad_idx     = 0
    ):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn_fwd    = CustomRNN(input_size=embed_dim, hidden_size=hidden_dim, recepie=recipe)
        self.rnn_bwd    = CustomRNN(input_size=embed_dim, hidden_size=hidden_dim, recepie=recipe)
        self.ln_fwd     = nn.LayerNorm(hidden_dim)
        self.ln_bwd     = nn.LayerNorm(hidden_dim)
        self.dropout    = nn.Dropout(dropout)
        # mean+max from fwd (hidden*2) concatenated with mean+max from bwd (hidden*2)
        self.classifier = nn.Linear(hidden_dim * 4, num_classes)

    def _run_cell(self, rnn, ln, emb_seq):
        """
        Run one CustomRNN cell over a full sequence with per-step LayerNorm.
        emb_seq: (batch, seq_len, embed_dim)
        returns: (seq_len, batch, hidden_dim)
        """
        batch_size = emb_seq.size(0)
        seq_len    = emb_seq.size(1)

        # Initialise hidden state tuple
        hidden = tuple(
            torch.zeros(1, batch_size, self.hidden_dim, device=emb_seq.device)
            for _ in range(rnn.cell.hidden_tuple_size)
        )

        outputs = []
        for t in range(seq_len):
            inp          = emb_seq[:, t, :].unsqueeze(0)      # (1, B, embed)
            out, hidden  = rnn(inp, hidden)                    # out: (1, B, hidden)
            out_normed   = ln(out.squeeze(0)).unsqueeze(0)     # (1, B, hidden)
            outputs.append(out_normed)
            # Normalise carried hidden state too — prevents inter-step accumulation
            hidden = tuple(ln(h.squeeze(0)).unsqueeze(0) for h in hidden)

        return torch.cat(outputs, dim=0)   # (seq_len, B, hidden)

    def forward(self, x):
        # x: (B, seq_len)
        emb = self.dropout(self.embedding(x))   # (B, seq_len, embed)

        # Forward pass
        fwd = self._run_cell(self.rnn_fwd, self.ln_fwd, emb)    # (seq, B, hidden)
        # Backward pass: flip along seq dim, run, flip back
        bwd = self._run_cell(self.rnn_bwd, self.ln_bwd, torch.flip(emb, [1]))
        bwd = torch.flip(bwd, [0])                               # (seq, B, hidden)

        # (seq, B, hidden) → (B, seq, hidden)
        fwd = fwd.transpose(0, 1)
        bwd = bwd.transpose(0, 1)

        # Mean + Max pool over sequence for each direction
        fwd_mean = fwd.mean(dim=1)                               # (B, hidden)
        fwd_max, _ = fwd.max(dim=1)                              # (B, hidden)
        bwd_mean = bwd.mean(dim=1)
        bwd_max, _ = bwd.max(dim=1)

        pooled = torch.cat([fwd_mean, fwd_max, bwd_mean, bwd_max], dim=1)  # (B, hidden*4)
        return self.classifier(self.dropout(pooled))


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ── Instantiate ──────────────────────────────────────────────────
nas_model = NASClassifier(
    vocab_size  = VOCAB_SIZE,
    recipe      = BEST_RECIPE,
    embed_dim   = 128,
    hidden_dim  = 256,
    num_classes = 2,
    dropout     = 0.3,
    pad_idx     = PAD_IDX
).to(DEVICE)

print(f'NAS parameters : {count_parameters(nas_model):,}')
print(f'Recipe nodes   : {list(BEST_RECIPE.keys())}')


## 6. Training Functions


In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss      = 0.0
    valid_batches   = 0
    skipped         = 0
    all_preds, all_labels = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss   = criterion(logits, y)

        if not torch.isfinite(loss):
            skipped += 1
            if skipped > 5:
                raise RuntimeError(
                    f'{skipped} non-finite losses in a row. '
                    f'Check Cell 4 printed activation ops — if empty, '
                    f're-run NB04 v9 to regenerate best_architecture.json.'
                )
            continue

        skipped = 0
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss  += loss.item()
        valid_batches += 1
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    if valid_batches == 0:
        raise RuntimeError('Every batch produced non-finite loss — recipe has no activation ops.')

    return (
        total_loss / valid_batches,
        accuracy_score(all_labels, all_preds),
        f1_score(all_labels, all_preds, average='macro')
    )


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss   = criterion(logits, y)
            if torch.isfinite(loss):
                total_loss += loss.item()
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    return {
        'loss'      : total_loss / len(loader),
        'accuracy'  : accuracy_score(all_labels, all_preds),
        'f1_macro'  : f1_score(all_labels, all_preds, average='macro'),
        'f1_off'    : f1_score(all_labels, all_preds, average='binary', pos_label=1),
        'precision' : precision_score(all_labels, all_preds, average='macro', zero_division=0),
        'recall'    : recall_score(all_labels, all_preds, average='macro', zero_division=0),
        'preds'     : all_preds,
        'labels'    : all_labels
    }


def train_model(model, model_name, train_loader, test_loader, device, epochs=30, lr=3e-4):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3
    )
    best_f1, best_metrics, history = 0.0, {}, []

    print(f'\n{"="*55}')
    print(f'  Training {model_name}')
    print(f'  lr={lr} | clip=1.0 | LayerNorm | mean+max pool | patience=3')
    print(f'{"="*55}')

    for epoch in range(1, epochs + 1):
        train_loss, train_acc, train_f1 = train_epoch(
            model, train_loader, optimizer, criterion, device
        )
        metrics = evaluate(model, test_loader, criterion, device)
        scheduler.step(metrics['f1_macro'])

        history.append({
            'epoch': epoch, 'train_loss': train_loss,
            'train_acc': train_acc, 'train_f1': train_f1,
            'test_loss': metrics['loss'], 'test_acc': metrics['accuracy'],
            'test_f1': metrics['f1_macro'],
        })

        print(
            f'Epoch {epoch:02d}/{epochs} | '
            f'Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | '
            f'Test Loss: {metrics["loss"]:.4f} | Test F1: {metrics["f1_macro"]:.4f} | '
            f'Test Acc: {metrics["accuracy"]:.4f}'
        )

        if metrics['f1_macro'] > best_f1:
            best_f1      = metrics['f1_macro']
            best_metrics = metrics.copy()
            torch.save(model.state_dict(),
                       os.path.join(MODELS_PATH, f'{model_name}_best.pt'))
            print(f'  ✓ New best F1: {best_f1:.4f} — model saved')

    return best_metrics, history

print('Training functions defined')


## 7. Train

In [ ]:
EPOCHS = 30
NAS_LR = 3e-4   # Lower than baseline 1e-3: gating-free cells are more LR-sensitive

nas_model = NASClassifier(
    vocab_size  = VOCAB_SIZE,
    recipe      = BEST_RECIPE,
    embed_dim   = 128,
    hidden_dim  = 256,
    num_classes = 2,
    dropout     = 0.3,
    pad_idx     = PAD_IDX
).to(DEVICE)

print(f'NAS parameters : {count_parameters(nas_model):,}')

nas_best, nas_history = train_model(
    nas_model, 'NAS_arch',
    train_loader, test_loader,
    DEVICE, epochs=EPOCHS, lr=NAS_LR
)


## 8. Results

In [ ]:
# Load baseline results from notebook 01
with open(os.path.join(RESULTS_PATH, 'baseline_results.json'), 'r') as f:
    baseline_results = json.load(f)

lstm_results = baseline_results['LSTM']
gru_results  = baseline_results['GRU']

print('\n' + '='*65)
print('  FINAL COMPARISON — SOLD Test Set')
print('='*65)
print(f'{"Model":<20} {"Accuracy":>10} {"F1 Macro":>10} {"F1 OFF":>10} {"Precision":>10} {"Recall":>10}')
print('-'*65)

models = [
    ('LSTM (baseline)',  lstm_results),
    ('GRU  (baseline)',  gru_results),
    ('NAS Architecture', nas_best),
]

for name, m in models:
    print(
        f'{name:<20} '
        f'{m["accuracy"]:>10.4f} '
        f'{m["f1_macro"]:>10.4f} '
        f'{m["f1_off"]:>10.4f} '
        f'{m["precision"]:>10.4f} '
        f'{m["recall"]:>10.4f}'
    )

print('='*65)

# Highlight winner
f1_scores = {
    'LSTM'           : lstm_results['f1_macro'],
    'GRU'            : gru_results['f1_macro'],
    'NAS Architecture': nas_best['f1_macro']
}
winner = max(f1_scores, key=f1_scores.get)
print(f'\n🏆 Best model by F1 Macro: {winner} ({f1_scores[winner]:.4f})')

## 9. Detailed Classification Report

In [ ]:
print('\n── NAS Architecture Classification Report ──')
print(classification_report(
    nas_best['labels'], nas_best['preds'],
    target_names=['NOT', 'OFF']
))

## 10. Training Curve Plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

all_histories = [
    (baseline_results['LSTM']['history'], 'LSTM'),
    (baseline_results['GRU']['history'],  'GRU'),
    (nas_history,                          f'NAS (arch {BEST_INDEX})'),
]

for ax, (history, name) in zip(axes, all_histories):
    epochs_x   = [h['epoch']    for h in history]
    train_f1   = [h['train_f1'] for h in history]
    test_f1    = [h['test_f1']  for h in history]

    ax.plot(epochs_x, train_f1, label='Train F1', marker='o', linewidth=2)
    ax.plot(epochs_x, test_f1,  label='Test F1',  marker='s', linewidth=2)
    ax.set_title(f'{name}', fontsize=13)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Macro F1')
    ax.set_ylim(0.4, 1.0)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Training Curves — SOLD Offensive Language Detection', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_PATH, 'all_training_curves.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()
print('Plot saved to results/all_training_curves.png')

## 11. Save Final Comparison

In [ ]:
final_comparison = {
    'LSTM': {
        'accuracy'  : lstm_results['accuracy'],
        'f1_macro'  : lstm_results['f1_macro'],
        'f1_off'    : lstm_results['f1_off'],
        'precision' : lstm_results['precision'],
        'recall'    : lstm_results['recall'],
    },
    'GRU': {
        'accuracy'  : gru_results['accuracy'],
        'f1_macro'  : gru_results['f1_macro'],
        'f1_off'    : gru_results['f1_off'],
        'precision' : gru_results['precision'],
        'recall'    : gru_results['recall'],
    },
    'NAS_Architecture': {
        'accuracy'       : nas_best['accuracy'],
        'f1_macro'       : nas_best['f1_macro'],
        'f1_off'         : nas_best['f1_off'],
        'precision'      : nas_best['precision'],
        'recall'         : nas_best['recall'],
        'arch_index'     : BEST_INDEX,
        'hidden_covariance': BEST_HC,
        'history'        : nas_history
    },
    'winner': winner
}

out_path = os.path.join(RESULTS_PATH, 'final_comparison.json')
with open(out_path, 'w') as f:
    json.dump(final_comparison, f, indent=2)

print(f'Final comparison saved to {out_path}')
print(f'\nThese results are ready to report in your CHiPSAL paper.')